In [2]:
from fileformer import utils, ByteLevelTokenizer, eval, CompressionEngine, FileFormer, FileDataset
import torch
from torch.profiler import profile, ProfilerActivity, record_function

In [3]:
tokenizer = ByteLevelTokenizer()
configs = utils.load_config("../configs/config.yml")
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=1)

In [4]:
fileformer = FileFormer.load_from_checkpoint("/Users/daniilogorodnikov/model-epoch=02-step=110000.ckpt", loss_fn=loss_fn, config=configs, device="cpu")

/Users/daniilogorodnikov/anaconda3/envs/notus/lib/python3.12/site-packages/lightning/pytorch/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.5, which is newer than your current Lightning version: v2.5.0.post0


In [5]:
from torch.utils.data import DataLoader
dataset = FileDataset(**configs['dataset'])
dataloader = DataLoader(dataset, batch_size=1,
                        shuffle=True, num_workers=configs['train']['num_workers']
                        )

Preparing files: 100%|██████████| 15/15 [00:00<00:00, 590.92it/s]

Ошибка обработки файла /Users/daniilogorodnikov/dataset/test/.DS_Store: 'NoneType' object cannot be interpreted as an integer


In [6]:
tokens, masked_tokens, pads, hash, extention_tokenize = next(iter(dataloader))

In [16]:
with profile(activities=[ProfilerActivity.CPU], profile_memory=True) as prof:
    with record_function("model_inference"):
        fileformer.forward(masked_tokens, hash, extention_tokenize)

In [17]:
print(prof.key_averages().table(sort_by="cpu_time_total"))

---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                       Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
---------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
            model_inference         5.64%      10.663ms       100.00%     189.095ms     189.095ms           0 B    -282.29 MB             1  
               aten::linear         0.11%     207.794us        64.76%     122.457ms       1.441ms      48.54 MB           0 B            85  
                aten::addmm        61.65%     116.583ms        63.82%     120.678ms       1.420ms      48.54 MB      48.54 MB            85  
              aten::dropout         0.07%     135.296us        13.68%      25.862ms     680.576us      56.60 MB           0 B            38  
      

In [15]:
prof.export_chrome_trace("trace.json")